In [3]:
!unzip /content/REES.zip

Archive:  /content/REES.zip
  inflating: zzzbruh/resume_classic.pdf  
  inflating: zzzbruh/resume_minimalist.pdf  
  inflating: zzzbruh/resume_modern_sections.pdf  
  inflating: zzzbruh/resume_table_multiline.pdf  
  inflating: zzzbruh/resume_two_column.pdf  
  inflating: zzzbruh/Sarvam_Jha_Artificial Intelligence.pdf  
  inflating: zzzbruh/yodolegend.pdf  


In [6]:
# ------------------------
# 1. Install & Imports
# ------------------------
!pip install pymupdf sentence-transformers spacy pandas tqdm openpyxl

import os, re, string, zipfile, tempfile
import pandas as pd
from tqdm import tqdm
import spacy
from sentence_transformers import SentenceTransformer, util
import fitz  # PyMuPDF

# Load models once
nlp = spacy.load("en_core_web_sm")
model = SentenceTransformer('all-MiniLM-L6-v2')


# ------------------------
# 2. Resume Text Extraction (Structured)
# ------------------------
def extract_resume_text(pdf_path):
    text = ""
    doc = fitz.open(pdf_path)

    for page in doc:
        blocks = page.get_text("dict")["blocks"]

        for block in blocks:
            if "lines" not in block:
                continue

            for line in block["lines"]:
                line_text = ""
                for span in line["spans"]:
                    line_text += span["text"]
                text += line_text.strip() + "\n"

        text += "\n"

    return text


# ------------------------
# 3. Clean Layout Noise
# ------------------------
def clean_layout_noise(text):
    lines = text.split("\n")
    clean_lines = []

    for line in lines:
        line = line.strip()
        if len(line) < 2:
            continue
        if line.count("  ") > 5:
            continue
        clean_lines.append(line)

    return "\n".join(clean_lines)


# ------------------------
# 4. Debug Print
# ------------------------
def print_resume_content(text):
    print("\n" + "="*60)
    print("📄 EXTRACTED RESUME TEXT")
    print("="*60 + "\n")
    print(text)
    print("\n" + "="*60 + "\n")


# ------------------------
# 5. Text Preprocessing
# ------------------------
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop]
    return " ".join(tokens)


# ------------------------
# 6. Keyphrase Score
# ------------------------
def keyphrase_score(text, skills):
    hits = sum(1 for s in skills if s.lower() in text)
    return hits / len(skills) if skills else 0


# ------------------------
# 7. Semantic Similarity Score
# ------------------------
def semantic_score(resume_text, job_description):
    emb_resume = model.encode(resume_text, convert_to_tensor=True)
    emb_job = model.encode(job_description, convert_to_tensor=True)
    sim = util.pytorch_cos_sim(emb_resume, emb_job)
    return float(sim)


# ------------------------
# 8. Combined Score
# ------------------------
def combined_score(resume_path, job_description, skills, debug_print=False):

    raw_text = extract_resume_text(resume_path)
    raw_text = clean_layout_noise(raw_text)

    if debug_print:
        print_resume_content(raw_text)

    clean_resume = preprocess_text(raw_text)
    clean_job = preprocess_text(job_description)

    k_score = keyphrase_score(clean_resume, skills)
    s_score = semantic_score(clean_resume, clean_job)

    final = 0.4 * k_score + 0.6 * s_score

    return final, k_score, s_score, raw_text


# ------------------------
# 9. Info Extraction
# ------------------------
def extract_emails(text):
    return re.findall(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', text)

def extract_phone_numbers(text):
    return re.findall(r'(\+?\d[\d \-\(\)]{9,}\d)', text)

def extract_name(text):
    doc = nlp(text)
    names = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
    return names[0] if names else None


# ------------------------
# 10. Collect Resume Files (NEW MULTI INPUT SUPPORT)
# ------------------------
def collect_resume_files(input_path):
    """
    Accepts:
    - Folder path
    - Single PDF
    - ZIP file
    - List of PDFs
    Returns list of PDF paths
    """

    pdf_files = []

    # If list
    if isinstance(input_path, list):
        return input_path

    # If single PDF
    if input_path.lower().endswith(".pdf"):
        return [input_path]

    # If ZIP file
    if input_path.lower().endswith(".zip"):
        temp_dir = tempfile.mkdtemp()

        with zipfile.ZipFile(input_path, 'r') as zip_ref:
            zip_ref.extractall(temp_dir)

        for root, _, files in os.walk(temp_dir):
            for f in files:
                if f.lower().endswith(".pdf"):
                    pdf_files.append(os.path.join(root, f))

        return pdf_files

    # If folder
    if os.path.isdir(input_path):
        for f in os.listdir(input_path):
            if f.lower().endswith(".pdf"):
                pdf_files.append(os.path.join(input_path, f))
        return pdf_files

    return []


# ------------------------
# 11. Analyze Multiple Resumes
# ------------------------
def analyze_resumes(input_path, job_description, skills, debug_print=False):

    results = []
    pdf_files = collect_resume_files(input_path)

    if not pdf_files:
        print("⚠️ No PDF resumes found.")
        return pd.DataFrame()

    for pdf_path in tqdm(pdf_files, desc="Analyzing resumes", colour="cyan"):

        try:
            final, k, s, raw_text = combined_score(
                pdf_path,
                job_description,
                skills,
                debug_print=debug_print
            )

            name = extract_name(raw_text)
            emails = extract_emails(raw_text)
            phones = extract_phone_numbers(raw_text)

            results.append({
                "Filename": os.path.basename(pdf_path),
                "Name": name,
                "Emails": ", ".join(emails) if emails else "",
                "Phones": ", ".join(phones) if phones else "",
                "Keyphrase_Score": round(k, 3),
                "Semantic_Score": round(s, 3),
                "Final_Score": round(final, 3)
            })

        except Exception as e:
            print(f"⚠️ Error processing {pdf_path}: {e}")

    df = pd.DataFrame(results)

    if not df.empty:
        df.sort_values(by="Final_Score", ascending=False, inplace=True)
        df.reset_index(drop=True, inplace=True)

    return df


# ------------------------
# 12. Example Usage
# ------------------------
if __name__ == "__main__":

    # Change this to:
    # - folder path
    # - zip file
    # - single pdf
    # - list of pdfs
    input_source = "/content/zzzbruh"

    job_desc = "We are looking for a Python developer with NLP and SQL experience."
    required_skills = ["natural language processing","machine learning"]

    df_results = analyze_resumes(
        input_source,
        job_desc,
        required_skills,
        debug_print=True
    )

    print("\n=== 📊 Resume Matching Results ===")
    display(df_results)

    save_path = "/content/resume_match_results.xlsx"
    df_results.to_excel(save_path, index=False)

    print(f"\n✅ Results saved to {save_path}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Analyzing resumes:  14%|█▍        | 1/7 [00:00<00:00,  6.98it/s]


📄 EXTRACTED RESUME TEXT

Michael Brown
Profile
Results-driven developer with 5+ years experience.
Skills
Java, Spring Boot, Docker, AWS
Projects
E-commerce platform serving 10k+ users daily.



📄 EXTRACTED RESUME TEXT

David Wilson
Key Achievements
Led a team of 5 engineers
Reduced deployment time by 40%
Implemented CI/CD pipelines




Analyzing resumes:  43%|████▎     | 3/7 [00:00<00:00, 10.10it/s]


📄 EXTRACTED RESUME TEXT

Name
Institute
Board/University
Skills
% / CGPA
Sarvam jha
IIT AGARTHA
NMIMS Mumbai
Natural language
processing
Machine learning
2.24
AGE-10000
PACE Junior science college
HSC
2021
80
Maneckji cooper education trust
(M.C.E.T)
ICSE
2019
92.6



📄 EXTRACTED RESUME TEXT

Emily Davis
Year
Role
Details
2022-2024
Frontend Dev
Worked on React apps
Improved UX significantly
2020-2022
Intern
Assisted in UI design
Automated testing tasks



📄 EXTRACTED RESUME TEXT

John Doe
Email: john@example.com | Phone: +1 234 567 890
Experience
Software Engineer - ABC Corp (2020-2024)
- Developed backend systems in Python
- Improved performance by 30%
Education
B.Sc. in Computer Science - XYZ University




...


📄 EXTRACTED RESUME TEXT

Jane Smith
Skills
Experience
Python, SQL, ML
Data Analyst - DataCorp (2019-2024)
Power BI, Excel
Business Analyst - Insight Ltd (2017-2019)



📄 EXTRACTED RESUME TEXT

EDUCATION
CORE SKILLS
INTERNSHIP
EXPERIENCE
SARVAM JHA
MBA Tech: Class of 2028
Technology Stream: Artificial Intelligence
Date of Birth: 05-December-2003
Qualification
Institute
Board/University
Year
% / CGPA
MBA Tech
Mukesh Patel School of Technology
Management and Engineering
NMIMS Mumbai
Currently pursuing
CGPA -2.24
SEM5 GPA-3.07
XII
PACE Junior science college
HSC
2021
80
Maneckji cooper education trust
(M.C.E.T)
ICSE
2019
92.6
● Python
● Preparing Deep learning models using neural networks
● Natural Language processing (NLP)
● Coursework Algorithms related to AI/ML like naïve bayes, swarm optimization, ant
colony optimization, k-means, alpha-beta pruning etc.
Community Service:
Spark A Change
PROFESSIONAL
ATTRIBUTES/
ACHIEVEMENTS
PROJECTS:
Face mask detector using convolution neural networ

Analyzing resumes: 100%|██████████| 7/7 [00:00<00:00,  8.37it/s]


=== 📊 Resume Matching Results ===


,Filename,Name,Emails,Phones,Keyphrase_Score,Semantic_Score,Final_Score
0,yodolegend.pdf,None,,,0.5,0.295,0.377
1,Sarvam_Jha_Artificial Intelligence.pdf,Coursework Algorithms,,,0.5,0.245,0.347
2,resume_two_column.pdf,Jane Smith,,,0.0,0.531,0.318
3,resume_classic.pdf,John Doe\nEmail,john@example.com,+1 234 567 890,0.0,0.308,0.185
4,resume_table_multiline.pdf,Emily Davis,,,0.0,0.156,0.093
5,resume_minimalist.pdf,David Wilson\nKey Achievements,,,0.0,0.087,0.052
6,resume_modern_sections.pdf,Michael Brown,,,0.0,0.085,0.051



✅ Results saved to /content/resume_match_results.xlsx
